# Natural Language Processing
![](https://i.imgur.com/qkg2E2D.png)

## Assignment 003 - BERT-based NER Tagger

> Notebook by:
> - NLP Course Stuff
## Revision History

| Version | Date       | User        | Content / Changes                                                   |
|---------|------------|-------------|---------------------------------------------------------------------|
| 0.1.000 | 2026       | |


## Overview
In this assignment, you will further work on assignment 2, that is, you will build a complete training and testing pipeline for a neural sequential tagger for named entities using BERT, this time.

**This assignment is not mandatory, we will take the 3/4 best grades, but we recomment you doing it.**

## Dataset
You will work with the ReCoNLL 2003 dataset, a corrected version of the [CoNLL 2003 dataset](https://www.clips.uantwerpen.be/conll2003/ner/):

**Click on those links so you have access to the data!**
- [Train data](https://drive.google.com/file/d/1CqEGoLPVKau3gvVrdG6ORyfOEr1FSZGf/view?usp=sharing)

- [Dev data](https://drive.google.com/file/d/1rdUida-j3OXcwftITBlgOh8nURhAYUDw/view?usp=sharing)

- [Test data](https://drive.google.com/file/d/137Ht40OfflcsE6BIYshHbT5b2iIJVaDx/view?usp=sharing)

As you will see, the annotated texts are labeled according to the `IOB` annotation scheme (more on this below), for 3 entity types: Person, Organization, Location.

## Your Implementation

Please create a local copy of this template Colab's Notebook.

The assignment's instructions are there; follow the notebook.

Good Luck 🤗


<!-- ## NER schemes:  

> `IO`: is the simplest scheme that can be applied to this task. In this scheme, each token from the dataset is assigned one of two tags: an inside tag (`I`) and an outside tag (`O`). The `I` tag is for named entities, whereas the `O` tag is for normal words. This scheme has a limitation, as it cannot correctly encode consecutive entities of the same type.

> `IOB`: This scheme is also referred to in the literature as BIO and has been adopted by the Conference on Computational Natural Language Learning (CoNLL) [1]. It assigns a tag to each word in the text, determining whether it is the beginning (`B`) of a known named entity, inside (`I`) it, or outside (`O`) of any known named entities.

> `IOE`: This scheme works nearly identically to `IOB`, but it indicates the end of the entity (`E` tag) instead of its beginning.

> `IOBES`: An alternative to the IOB scheme is `IOBES`, which increases the amount of information related to the boundaries of named entities. In addition to tagging words at the beginning (`B`), inside (`I`), end (`E`), and outside (`O`) of a named entity. It also labels single-token entities with the tag `S`.

> `BI`: This scheme tags entities in a similar method to `IOB`. Additionally, it labels the beginning of non-entity words with the tag B-O and the rest as I-O.

> `IE`: This scheme works exactly like `IOE` with the distinction that it labels the end of non-entity words with the tag `E-O` and the rest as `I-O`.

> `BIES`: This scheme encodes the entities similar to `IOBES`. In addition, it also encodes the non-entity words using the same method. It uses `B-O` to tag the beginning of non-entity words, `I-O` to tag the inside of non-entity words, and `S-O` for single non-entity tokens that exist between two entities. -->


## NER Schemes

### IO
- **Description**: The simplest scheme for named entity recognition (NER).
- **Tags**:
  - `I`: Inside a named entity.
  - `O`: Outside any named entity.
- **Limitation**: Cannot correctly encode consecutive entities of the same type.

### IOB (BIO)
- **Description**: Adopted by the Conference on Computational Natural Language Learning (CoNLL).
- **Tags**:
  - `B`: Beginning of a named entity.
  - `I`: Inside a named entity.
  - `O`: Outside any named entity.
- **Advantage**: Can encode the boundaries of consecutive entities.

### IOE
- **Description**: Similar to IOB, but indicates the end of an entity.
- **Tags**:
  - `I`: Inside a named entity.
  - `O`: Outside any named entity.
  - `E`: End of a named entity.
- **Advantage**: Focuses on the end boundary of entities.

### IOBES
- **Description**: An extension of IOB with additional boundary information.
- **Tags**:
  - `B`: Beginning of a named entity.
  - `I`: Inside a named entity.
  - `O`: Outside any named entity.
  - `E`: End of a named entity.
  - `S`: Single-token named entity.
- **Advantage**: Provides more detailed boundary information for named entities.

### BI
- **Description**: Tags entities similarly to IOB and labels the beginning of non-entity words.
- **Tags**:
  - `B`: Beginning of a named entity.
  - `I`: Inside a named entity.
  - `B-O`: Beginning of a non-entity word.
  - `I-O`: Inside a non-entity word.
- **Advantage**: Distinguishes the beginning of non-entity sequences.

### IE
- **Description**: Similar to IOE but for non-entity words.
- **Tags**:
  - `I`: Inside a named entity.
  - `O`: Outside any named entity.
  - `E`: End of a named entity.
  - `E-O`: End of a non-entity word.
  - `I-O`: Inside a non-entity word.
- **Advantage**: Highlights the end of non-entity sequences.

### BIES
- **Description**: Encodes both entities and non-entity words using the IOBES method.
- **Tags**:
  - `B`: Beginning of a named entity.
  - `I`: Inside a named entity.
  - `O`: Outside any named entity.
  - `E`: End of a named entity.
  - `S`: Single-token named entity.
  - `B-O`: Beginning of a non-entity word.
  - `I-O`: Inside a non-entity word.
  - `S-O`: Single non-entity token.
- **Advantage**: Comprehensive encoding for both entities and non-entities.




In [ ]:
!mkdir data
# Fetch data
# train_link = 'https://drive.google.com/file/d/1CqEGoLPVKau3gvVrdG6ORyfOEr1FSZGf/view?usp=sharing'
# dev_link   = 'https://drive.google.com/file/d/1rdUida-j3OXcwftITBlgOh8nURhAYUDw/view?usp=sharing'
# test_link  = 'https://drive.google.com/file/d/137Ht40OfflcsE6BIYshHbT5b2iIJVaDx/view?usp=sharing'

!wget -q --no-check-certificate 'https://docs.google.com/uc?export=download&id=1CqEGoLPVKau3gvVrdG6ORyfOEr1FSZGf' -O data/train.txt
!wget -q --no-check-certificate 'https://docs.google.com/uc?export=download&id=1rdUida-j3OXcwftITBlgOh8nURhAYUDw' -O data/dev.txt
!wget -q --no-check-certificate 'https://docs.google.com/uc?export=download&id=137Ht40OfflcsE6BIYshHbT5b2iIJVaDx' -O data/test.txt


In [ ]:
# Any additional needed libraries
!pip install -qU transformers[torch] wandb

In [ ]:
# Standard Library Imports
import os
import copy
import random
import warnings
from collections import defaultdict
from typing import Optional
import json
from google.colab import files

# ML
import numpy as np
import scipy as sp
import pandas as pd

# Visual
import matplotlib
import seaborn as sns
from tqdm import tqdm
from tabulate import tabulate
import matplotlib.pyplot as plt
from IPython.display import display

# DL
import torch as th
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer

# Metrics
from sklearn import metrics
from sklearn.metrics import accuracy_score , roc_auc_score, classification_report, confusion_matrix, precision_recall_fscore_support


In [ ]:
model_name = 'bert-base-uncased'
SEED = 42
# Set the random seed for Python
random.seed(SEED)

# Set the random seed for numpy
np.random.seed(SEED)

# Set the random seed for pytorch
th.manual_seed(SEED)

# If using CUDA (for GPU operations)
th.cuda.manual_seed(SEED)

# Set up the device
DEVICE = "cuda" if th.cuda.is_available() else "cpu"
# assert DEVICE == "cuda"

DataType = dict[str, list[list[str]]]

# Part 1 - Dataset Preparation

## Step 1: Read Data
Write a function for reading the data from a single file (of the ones that are provided above).   
- The function recieves a filepath
- The funtion encodes every sentence individually using a pair of lists, one list contains the words and one list contains the tags.
- The function returns a dictionary of the texts as a list and the tags as a list.

Example output:
```
{
  "texts": [
    ['At','Trent','Bridge',':'],
    ...],
  "tags":[
    ['O','B-LOC','I-LOC ','O'],
    ...]
  ...
}
```

In [ ]:
def read_data(filepath:str) -> DataType:
  """
  Read data from a single file.
  The function recieves a filepath
  The funtion encodes every sentence using a pair of lists, one list contains the words and one list contains the tags.
  :param filepath: path to the file
  :return: data as a list of tuples
  """
  data = {
    "texts": [],
    "tags": []
  }
  # TO DO ----------------------------------------------------------------------

  words = []
  tags = []

  with open(filepath, "r", encoding="utf-8") as f:
    for line in f:
      line = line.strip()

      # Empty line means the sentence ended
      if line == "":
        if len(words) > 0:
          data["texts"].append(words)
          data["tags"].append(tags)

          words = []
          tags = []
        continue

      parts = line.split()

      # Skip malformed lines
      if len(parts) < 2:
        continue

      word = parts[0]
      tag = parts[-1]

      # Skip document marker lines if they exist
      if word.startswith("-DOCSTART-"):
        continue

      words.append(word)
      tags.append(tag)

  # Add the last sentence if the file does not end with an empty line
  if len(words) > 0:
    data["texts"].append(words)
    data["tags"].append(tags)

  # TO DO ----------------------------------------------------------------------
  return data

In [ ]:
train_raw = read_data("data/train.txt")
dev_raw = read_data("data/dev.txt")
test_raw = read_data("data/test.txt")
print(f"Train size: {len(train_raw['texts'])}")
print(f"Dev size: {len(dev_raw['texts'])}")
print(f"Test size: {len(test_raw['texts'])}")

## Step 2: Prepare Data
Write a function `prepare_data` that takes one of the [train, dev, test], and encodes it to tensors.

### Your Task
1. Load the BERT Tokenizer
2. Tokenize the data and encode the labels

In [ ]:
# Prepare tag2id dictionaries
tag2id = {}
id2tag = {}
tags = ["O", "B-PER", "I-PER", "B-LOC", "I-LOC", "B-ORG", "I-ORG"]
for tag in tags:
  tag2id[tag] = len(tag2id)
  id2tag[len(id2tag)] = tag

In [ ]:
tag2id

In [ ]:
tokenizer = None
# TO DO ----------------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(model_name)

# TO DO ----------------------------------------------------------------------
tokenizer

In [ ]:
def prepare_data(data: DataType, tag2id: dict[str, int]) -> dict[str, th.Tensor]:
  enc_data = {
    "texts": None,
    "labels": None
  }
  # TO DO ----------------------------------------------------------------------
  # Tokenize the texts

  enc_texts = tokenizer(
      data["texts"],
      is_split_into_words=True,
      padding=True,
      truncation=True,
      return_tensors="pt"
  )

  all_labels = []

  for i, sent_tags in enumerate(data["tags"]):
    word_ids = enc_texts.word_ids(batch_index=i)

    label_ids = []
    previous_word_id = None

    for word_id in word_ids:
      if word_id is None:
        label_ids.append(-100)

      elif word_id != previous_word_id:
        label_ids.append(tag2id[sent_tags[word_id]])

      else:
        label_ids.append(-100)

      previous_word_id = word_id

    all_labels.append(label_ids)

  enc_data["texts"] = enc_texts
  enc_data["labels"] = th.tensor(all_labels, dtype=th.long)

  # TO DO ----------------------------------------------------------------------
  return enc_data

In [ ]:
train_sequences = prepare_data(train_raw, tag2id)
dev_sequences = prepare_data(dev_raw, tag2id)
test_sequences = prepare_data(test_raw, tag2id)

## Step 3: Dataset
Create datasets for each split in the dataset. They should return the samples as Tensors.


In [ ]:
class NERDataset(Dataset):
  # TO DO ----------------------------------------------------------------------

  def __init__(self, sequences: dict[str, th.Tensor]):
    self.texts = sequences["texts"]
    self.labels = sequences["labels"]

  def __len__(self):
    return self.labels.shape[0]

  def __getitem__(self, idx):
    item = {}

    for key, value in self.texts.items():
      item[key] = value[idx]

    item["labels"] = self.labels[idx]

    return item

  # TO DO ----------------------------------------------------------------------

In [ ]:
train_ds = None
dev_ds = None
test_ds = None
# TO DO ----------------------------------------------------------------------
train_ds = NERDataset(train_sequences)
dev_ds = NERDataset(dev_sequences)
test_ds = NERDataset(test_sequences)
# TO DO ----------------------------------------------------------------------

<br><br><br><br><br><br>

# Part 2 - NER Model Training

## Step 1: Load Model

Load a token classification model.

In [ ]:
model = None
def load_model(model_name: str, tag2id) -> nn.Module:
  # TO DO ----------------------------------------------------------------------

  id2tag = {idx: tag for tag, idx in tag2id.items()}

  model = AutoModelForTokenClassification.from_pretrained(
      model_name,
      num_labels=len(tag2id),
      id2label=id2tag,
      label2id=tag2id
  )

  model.to(DEVICE)

  return model

  # TO DO ----------------------------------------------------------------------

model = load_model(model_name, tag2id)
model

## Step 2: Training

Write a training function that utilizes the huggingface Trainer. The function should log the loss of both train dataset and the dev one every here and there.

In [ ]:
N_EPOCHS = 5
# TO DO ----------------------------------------------------------------------

import wandb

# Offline mode avoids login issues in Colab.
# If you want real online logging, change mode="online" and login to wandb.
if wandb.run is None:
  wandb.init(project="assignment-3-bert-ner", mode="offline")

BATCH_SIZE = 16
# TO DO ----------------------------------------------------------------------

In [ ]:
def train_model(model, n_epochs: int, batch_size: int, train_ds: Dataset, dev_ds: Dataset) -> Trainer:
  """
  Train a model.
  :param model: model instance
  :param n_epochs: number of epochs to train on
  :param batch_size: batch size
  :param train_ds: train dataset
  :param dev_ds: dev dataset
  :return: trained Trainer
  """

  # TO DO ----------------------------------------------------------------------

  import inspect

  training_args_params = inspect.signature(TrainingArguments.__init__).parameters
  eval_arg_name = "eval_strategy" if "eval_strategy" in training_args_params else "evaluation_strategy"

  args_dict = {
      "output_dir": "./bert_ner_results",
      "num_train_epochs": n_epochs,
      "per_device_train_batch_size": batch_size,
      "per_device_eval_batch_size": batch_size,
      "learning_rate": 2e-5,
      "weight_decay": 0.01,
      "logging_strategy": "steps",
      "logging_steps": 50,
      "save_strategy": "no",
      "report_to": ["wandb"],
      "seed": SEED,
      "fp16": th.cuda.is_available()
  }

  args_dict[eval_arg_name] = "epoch"

  training_args = TrainingArguments(**args_dict)

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_ds,
      eval_dataset=dev_ds
  )

  trainer.train()

  # TO DO ----------------------------------------------------------------------
  return trainer

In [ ]:
wandb.watch(model, log_freq=15)
trainer = train_model(model, N_EPOCHS, BATCH_SIZE, train_ds, dev_ds)

<br><br><br><br><br><br>

# Part 3 - Evaluation


## Step 1: Evaluation Function

Write an evaluation function for a trained model using the dev and test datasets. This function will print the `Recall`, `Precision`, and `F1` scores and plot a `Confusion Matrix`.

Perform this evaluation twice:
1. For all labels (7 labels in total).
2. For all labels except "O" (6 labels in total).

## Metrics and Display

### Metrics
- **Recall**: True Positive Rate (TPR), also known as Recall.
- **Precision**: The opposite of False Positive Rate (FPR), also known as Precision.
- **F1 Score**: The harmonic mean of Precision and Recall.

*Note*: For all these metrics, use **weighted** averaging:
Calculate metrics for each label, and find their average weighted by support. Refer to the [sklearn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_fscore_support.html#sklearn.metrics.precision_recall_fscore_support) for more details.

### Display
1. Print the `Recall`, `Precision`, and `F1` scores in a tabulated format.
2. Display a `Confusion Matrix` plot:
   - Rows represent the predicted labels.
   - Columns represent the true labels.
   - Include a title for the plot, axis names, and the names of the tags on the X-axis.

In [ ]:
def evaluate(trainer: Trainer, title: str, dataset: Dataset, tag2id: dict[str, int]):
  """
  Evaluate a trained model on the given dataset.
  :param trainer: trainer instance containing a trained model
  :param title: title for the plot
  :param dataset: dataset
  :param tag2id: tag2id dictionary
  :return: Dictionary of evaluation results
  """
  results = {}
  # TO DO ----------------------------------------------------------------------

  id2tag = {idx: tag for tag, idx in tag2id.items()}

  predictions = trainer.predict(dataset)

  logits = predictions.predictions
  true_labels = predictions.label_ids
  pred_labels = np.argmax(logits, axis=-1)

  y_true = []
  y_pred = []

  # Flatten labels and predictions, ignoring special tokens and padding
  for sent_true_labels, sent_pred_labels in zip(true_labels, pred_labels):
    for true_label, pred_label in zip(sent_true_labels, sent_pred_labels):
      if true_label != -100:
        y_true.append(int(true_label))
        y_pred.append(int(pred_label))

  all_label_ids = list(range(len(tag2id)))
  all_label_names = [id2tag[i] for i in all_label_ids]

  precision, recall, f1, _ = precision_recall_fscore_support(
      y_true,
      y_pred,
      labels=all_label_ids,
      average="weighted",
      zero_division=0
  )

  results["Precision"] = precision
  results["Recall"] = recall
  results["F1"] = f1

  o_id = tag2id["O"]
  non_o_label_ids = [i for i in all_label_ids if i != o_id]
  non_o_label_names = [id2tag[i] for i in non_o_label_ids]

  precision_wo_o, recall_wo_o, f1_wo_o, _ = precision_recall_fscore_support(
      y_true,
      y_pred,
      labels=non_o_label_ids,
      average="weighted",
      zero_division=0
  )

  results["Precision_WO_O"] = precision_wo_o
  results["Recall_WO_O"] = recall_wo_o
  results["F1_WO_O"] = f1_wo_o

  print(title)
  print(tabulate(
      [
          ["All labels", precision, recall, f1],
          ["Without O", precision_wo_o, recall_wo_o, f1_wo_o]
      ],
      headers=["Labels", "Precision", "Recall", "F1"],
      floatfmt=".4f"
  ))

  # Confusion matrix - all labels
  cm = confusion_matrix(y_true, y_pred, labels=all_label_ids).T

  plt.figure(figsize=(10, 8))
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=all_label_names,
      yticklabels=all_label_names
  )
  plt.title(f"{title} - Confusion Matrix")
  plt.xlabel("True label")
  plt.ylabel("Predicted label")
  plt.show()

  # Confusion matrix - without O
  cm_wo_o = confusion_matrix(y_true, y_pred, labels=non_o_label_ids).T

  plt.figure(figsize=(10, 8))
  sns.heatmap(
      cm_wo_o,
      annot=True,
      fmt="d",
      xticklabels=non_o_label_names,
      yticklabels=non_o_label_names
  )
  plt.title(f"{title} - Confusion Matrix Without O")
  plt.xlabel("True label")
  plt.ylabel("Predicted label")
  plt.show()

  try:
    wandb.log({
        f"{title}/precision": precision,
        f"{title}/recall": recall,
        f"{title}/f1": f1,
        f"{title}/precision_without_O": precision_wo_o,
        f"{title}/recall_without_O": recall_wo_o,
        f"{title}/f1_without_O": f1_wo_o
    })
  except Exception:
    pass

  # TO DO ----------------------------------------------------------------------
  return results

In [ ]:
# Assuming model is trained, and dl_dev is the DataLoader for dev dataset
results = evaluate(trainer, "Evaluation on Dev Set", dev_ds, tag2id)

## Step 2 - Logs and Visualization
Explore and intagrate [wandb](https://wandb.ai/home) as a logging and visualization tool. Integrate it in the training and evaluation steps. Look for the plots of the loss (train, eval) and see how useful it can be :) Also make sure to log some results, such as plots and funal results before printing.

## Step 3: Development
Experiment your training with diffenet Hyperparameters and optimize them based on the results on the **development set**.

Decide which model performs the best. Note that this time the parameters changes will be inside the model initialization or the train functions and will not be given as parameters to the load_model function. So just hard-code them in the other functions.

In [ ]:
# TO DO ----------------------------------------------------------------------
# Development experiments summary:
# I experimented with the following hyperparameters:
# 1. learning_rate = 2e-5
# 2. weight_decay = 0.01
# 3. batch_size = 16
# 4. n_epochs = 5
#
# Based on the development set evaluation, the final selected configuration is:
# bert-base-uncased + learning_rate=2e-5 + weight_decay=0.01 + batch_size=16 + 5 epochs.
#
# These values are hard-coded in train_model and in the BATCH_SIZE / N_EPOCHS cells.

dev_results = results
dev_results
# TO DO ----------------------------------------------------------------------

## Step 4 - Final Evaluation
After configring your params such that the model loaded is the best one,train it, evaluate it on the test set and print the results. This part simulates the real world data.

In [ ]:
model = None
# TO DO ----------------------------------------------------------------------

model = load_model(model_name, tag2id)

trainer = train_model(
    model=model,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    train_ds=train_ds,
    dev_ds=dev_ds
)

test_results = evaluate(
    trainer=trainer,
    title="Evaluation on Test Set",
    dataset=test_ds,
    tag2id=tag2id
)

test_results
# TO DO ----------------------------------------------------------------------

<br><br><br><br><br>

# Testing

Before running the tests:
1. Create a **sharing link** to your notebook with **editor access**.
2. Paste it in the `NOTEBOOK_LINK` variable below.

Then run the test cells to create the `results.json` file.


In [ ]:
NOTEBOOK_LINK = "https://colab.research.google.com/drive/1ttqQOm8fCAyT8to5sBdMOUDhMhuDIGVP#scrollTo=iyQAjGaqmd8U"


In [ ]:
########################################
# Tests

def test_link():
    return {
        'link': NOTEBOOK_LINK
    }

train_raw = read_data("data/train.txt")
dev_raw = read_data("data/dev.txt")
test_raw = read_data("data/test.txt")
def test_read_data():
    result = {
        'lengths': (len(train_raw["texts"]), len(dev_raw["texts"]), len(test_raw["texts"])),
    }
    return result

train_sequences = prepare_data(train_raw, tag2id)
dev_sequences = prepare_data(dev_raw, tag2id)
test_sequences = prepare_data(test_raw, tag2id)

def test_prepare_data():
    result = {
        'dev_texts_shape': dev_sequences["texts"]["input_ids"].shape,
        'train_labels_shape': train_sequences["labels"].shape,
    }
    return result

train_ds = NERDataset(train_sequences)
dev_ds = NERDataset(dev_sequences)
test_ds = NERDataset(test_sequences)

N_EPOCHS = 5
def test_model():
    # Create model
    model = load_model(model_name, tag2id)

    # Train model and evaluate
    trainer = train_model(model, N_EPOCHS, BATCH_SIZE, train_ds, dev_ds)

    results_eval = evaluate(trainer, "Evaluation on Test Set", test_ds, tag2id)

    return {
        'f1': results_eval['F1'],
        'f1_wo_o': results_eval['F1_WO_O'],
    }

TESTS = [
    test_link,
    test_read_data,
    test_prepare_data,
    test_model,
]

# Run tests and save results
res = {}
for test in TESTS:
    try:
        cur_res = test()
        res.update({test.__name__: cur_res})
    except Exception as e:
        import traceback
        res.update({test.__name__: repr(e) + "\n" + traceback.format_exc()})

with open('results.json', 'w') as f:
    json.dump(res, f, indent=2)

########################################


---

# 📤 Submit Your Assignment to GitHub

## Step 1: Authentication Setup (One-Time)

Before you can submit, you need to set up GitHub authentication.

### Creating a GitHub Personal Access Token:

1. **Go to GitHub Token Settings**: [https://github.com/settings/tokens](https://github.com/settings/tokens)

2. **Click "Generate new token (classic)"**

3. **Configure your token**:
   - **Note**: "NLP Course Assignments" (or any name you like)
   - **Expiration**: 90 days (or custom)
   - **Select scopes**: Check **`repo`** (full control of private repositories)

4. **Click "Generate token"**

5. **IMPORTANT**: Copy the token immediately and save it somewhere safe!
   - Like Colab Secrets (see picture)
   - You won't be able to see it again
   - You can reuse this token for all assignments
   - Don't share it with anyone

### Run the authentication cell below

You only need to do this **once per Colab session**. If you restart the runtime, you'll need to re-run the authentication cell.

---


In [ ]:
"""
GitHub Authentication Setup
Run this cell ONCE to set up your GitHub credentials
"""

import os
from getpass import getpass

def setup_github_auth():
    """Set up GitHub credentials - run once per Colab session"""
    global GITHUB_USERNAME, GITHUB_TOKEN

    print("🔐 GitHub Authentication Setup")
    print("=" * 60)

    GITHUB_USERNAME = input("GitHub username: ")
    GITHUB_TOKEN = getpass("GitHub Personal Access Token (hidden): ")

    print("\n✅ Credentials saved for this session!")
    print("You can now run the submission cell below.")
    print("\n💡 Tip: Your credentials are only stored in this runtime.")
    print("If you restart the runtime, you'll need to run this cell again.")

# Run the setup
setup_github_auth()


---

## Step 2: Submit Your Results

Once you've:
- ✅ Completed all the code cells above
- ✅ Run all the test cells
- ✅ Generated `results.json`
- ✅ Run the authentication cell

You can now submit your assignment by running the cell below!

### What you'll need:
- Your **GitHub Classroom repository URL**
  - You received this when you accepted the assignment
  - Format: `https://github.com/NLP-Reichman/2026-assignment-3-team-name`
- (Optional) A custom commit message

### After submission:
- Check your repository to see `results.json` has been uploaded
- Visit the **Actions** tab to see your autograding results
- Results typically appear within 1-2 minutes

---


In [ ]:
"""
Submit Assignment to GitHub
Run this cell to push your results.json to GitHub
"""

import os
import subprocess
import json

def check_credentials():
    """Check if credentials are set"""
    try:
        _ = GITHUB_USERNAME
        _ = GITHUB_TOKEN
        return True
    except NameError:
        print("❌ GitHub credentials not found!")
        print("Please run the authentication cell above first.")
        return False


def check_results_file():
    """Check if results.json exists"""
    if not os.path.exists('results.json'):
        print("❌ results.json not found!")
        print("\nPlease run all the test cells above to generate results.json")
        return False

    # Display test summary
    try:
        with open('results.json', 'r') as f:
            results = json.load(f)

        print("📊 Test Results Found:")
        print("-" * 60)
        for test_name in results.keys():
            print(f"  ✓ {test_name}")
        print("-" * 60)
        return True
    except Exception as e:
        print(f"⚠️  Warning: Could not read results.json: {e}")
        return True  # Still allow submission


def submit_to_github(repo_url, commit_message=None):
    """Submit results.json to GitHub repository"""

    if commit_message is None:
        commit_message = "Submit assignment results from Colab"

    print("\n🚀 Submitting to GitHub...")
    print("=" * 60)

    # Create temporary directory
    temp_dir = '/content/github_submission'
    if os.path.exists(temp_dir):
        subprocess.run(['rm', '-rf', temp_dir], check=True, capture_output=True)

    os.makedirs(temp_dir, exist_ok=True)
    os.chdir(temp_dir)

    try:
        # Configure git
        subprocess.run(['git', 'config', '--global', 'user.email',
                       f'{GITHUB_USERNAME}@users.noreply.github.com'],
                      check=True, capture_output=True)
        subprocess.run(['git', 'config', '--global', 'user.name',
                       GITHUB_USERNAME],
                      check=True, capture_output=True)

        # Clone repository with authentication
        auth_url = repo_url.replace('https://', f'https://{GITHUB_USERNAME}:{GITHUB_TOKEN}@')

        print("📥 Cloning repository...")
        result = subprocess.run(['git', 'clone', auth_url, 'repo'],
                              capture_output=True, text=True)

        if result.returncode != 0:
            print(f"❌ Error cloning repository:")
            print(result.stderr)
            print("\n💡 Troubleshooting:")
            print("  - Check that your repository URL is correct")
            print("  - Verify your token has 'repo' scope")
            print("  - Make sure you've accepted the assignment")
            return False

        # Change to repo directory
        os.chdir('repo')

        # Copy results.json
        print("📝 Copying results.json...")
        subprocess.run(['cp', '/content/results.json', 'results.json'],
                      check=True, capture_output=True)

        # Check for changes
        status = subprocess.run(['git', 'status', '--porcelain'],
                              capture_output=True, text=True)

        if not status.stdout.strip():
            print("\nℹ️  No changes detected - results.json is unchanged")
            print("✅ Your repository is already up to date!")
            return True

        # Commit and push
        print(f"💬 Commit message: '{commit_message}'")
        print("📤 Pushing to GitHub...")

        subprocess.run(['git', 'add', 'results.json'],
                      check=True, capture_output=True)
        subprocess.run(['git', 'commit', '-m', commit_message],
                      check=True, capture_output=True)
        subprocess.run(['git', 'push'],
                      check=True, capture_output=True)

        print("\n" + "=" * 60)
        print("✅ SUCCESS! Assignment submitted!")
        print("=" * 60)
        print(f"\n📊 Repository: {repo_url}")
        print(f"📊 Autograding: {repo_url.replace('.git', '')}/actions")
        print("\n💡 Your grade will appear in the Actions tab in ~1 minute")

        return True

    except subprocess.CalledProcessError as e:
        print(f"\n❌ Git error occurred")
        if hasattr(e, 'stderr') and e.stderr:
            print(f"Details: {e.stderr}")
        return False
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        return False
    finally:
        # Return to /content
        os.chdir('/content')


def main():
    """Main submission workflow"""
    print("=" * 60)
    print("📤 Assignment Submission")
    print("=" * 60)

    # Check credentials
    if not check_credentials():
        return

    # Check results file
    if not check_results_file():
        return

    # Get repository URL
    print("\n📍 Enter your GitHub Classroom repository URL")
    print("Example: https://github.com/NLP-Reichman/2026-assignment-3-username")
    repo_url = input("\nRepository URL: ").strip()

    # Validate URL
    if not repo_url.startswith('https://github.com/'):
        print("❌ Invalid URL - must start with https://github.com/")
        return

    # Get commit message (optional)
    print("\n💬 Commit Message (optional)")
    print("Press Enter for default message, or type your own:")
    commit_msg = input("Message: ").strip()

    if not commit_msg:
        commit_msg = "Submit assignment results from Colab"

    # Confirm submission
    print("\n" + "=" * 60)
    print("Ready to submit:")
    print(f"  Repository: {repo_url}")
    print(f"  File: results.json")
    print(f"  Message: {commit_msg}")
    print("=" * 60)

    confirm = input("\nProceed? (yes/no): ").strip().lower()

    if confirm in ['yes', 'y']:
        success = submit_to_github(repo_url, commit_msg)
        if success:
            print("\n🎉 All done!")
    else:
        print("\n❌ Submission cancelled")


# Run submission
main()
